# New baselines, tier A -- centrality measures

Implements `TODO.md` item 4 (item 3, statistical rigor, skipped per request). Four classic,
citable `networkx` graph-centrality measures, added as new `file_type`s under
`baseline_candidates/`, following the exact `highest_degree`/`most_children` cell pattern
from `3.baseline_candidate_selection.ipynb`: rank every candidate node once, save the full
ranking, `.head(k)` at eval time.

- **PageRank** -- classic global importance measure; preempts the standard "why not just use
  PageRank" reviewer question.
- **Personalized PageRank seeded on `M`** -- biases the random-walk restart distribution
  toward the mapped concepts themselves, making it the fairest centrality baseline since,
  like the paper's own method, it's aware of `M` rather than purely structural.
- **Closeness centrality** -- conceptually the closest existing baseline to `distance_score`
  (ranks nodes by how close they are to everything else).
- **Eigenvector centrality** -- another standard, cheap-to-cite classical measure.

No new evaluation code here -- these are just new candidate-selection rankings. Re-run
`4.comparison_candidate_set.ipynb` afterwards to score them; its glob-based `all_candidates`
discovery (and `drilldown.load_all_candidates()` / `app_new_tokenizer.py`) picks up any new
`baseline_candidates/*.parquet` file automatically, no code changes needed there.

In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
import pickle

import networkx as nx
import polars as pl

import src.graph_tokenizer_gd_tree_dev.config as config

## Load graph, mapped concepts, id_to_label

In [ ]:
with open(config.ProcessedGraph().combined_subgraphs, "rb") as f:
    combined_subgraphs = pickle.load(f)
with open(config.ProcessedGraph().id_to_label, "rb") as f:
    id_to_label = pickle.load(f)

df_mapped = pl.read_parquet(config.BasicConfig().mapped_path)
mapped_ids = sorted(df_mapped["id"].unique().to_list())


def to_ranked_df(scores: dict, score_col: str) -> pl.DataFrame:
    """dict{node: score} -> the same token/score/label/index schema notebook 3's baselines use,
    sorted so .head(k) at eval time gives the top-k under this ranking."""
    return (
        pl.DataFrame({"token": list(scores.keys()), score_col: list(scores.values())})
        .with_columns(pl.col("token").replace_strict(id_to_label, default=None).alias("label"))
        .sort(score_col, descending=True)
        .with_row_index()
    )

## PageRank

`nx.pagerank` runs directly on the `MultiDiGraph` -- a `(u, v)` pair connected by two
different relation types gets more random-walk weight than a single-relation pair, which is
a reasonable reading of "more strongly connected," so the multi-edges are left as-is here
(unlike closeness/eigenvector below, which need a simplified graph for other reasons).

In [ ]:
pagerank = nx.pagerank(combined_subgraphs)
df_pagerank = to_ranked_df(pagerank, "pagerank")
df_pagerank.write_parquet(config.CandidateLists().pagerank)
df_pagerank.head()

## Personalized PageRank seeded on `M`

Same call, but the random walk restarts at a mapped concept instead of a uniformly random
node -- `personalization={m: 1 for m in mapped_ids}` (networkx normalizes this internally;
nodes absent from the dict get personalization value 0).

In [ ]:
personalized_pagerank = nx.pagerank(
    combined_subgraphs,
    personalization={m: 1 for m in mapped_ids if m in combined_subgraphs},
)
df_ppr = to_ranked_df(personalized_pagerank, "personalized_pagerank")
df_ppr.write_parquet(config.CandidateLists().personalized_pagerank)
df_ppr.head()

## Closeness centrality & eigenvector centrality

Both need a simple graph: `eigenvector_centrality` raises `NetworkXNotImplemented` outright
on a `MultiDiGraph`; `closeness_centrality` happens to run on one, but converting once and
reusing for both keeps them consistent with how `highest_degree`/`most_children` already
treat this graph -- ranking by *distinct* predecessors, not multi-edge occurrences between
the same pair.

For directed graphs, `nx.closeness_centrality`'s default direction is the node's *incoming*
distance -- "how close is everything else to reaching this node" -- which is exactly the
notion `highest_degree` already ranks by (distinct predecessors within `D` hops), just as a
continuous, whole-graph-shortest-path measure instead of a bounded-radius count. No
`G.reverse()` needed.

In [ ]:
G_simple = nx.DiGraph(combined_subgraphs)  # multi-edges collapsed, shared by both cells below

closeness_centrality = nx.closeness_centrality(G_simple)
df_closeness = to_ranked_df(closeness_centrality, "closeness_centrality")
df_closeness.write_parquet(config.CandidateLists().closeness_centrality)
df_closeness.head()

In [ ]:
eigenvector_centrality = nx.eigenvector_centrality(G_simple, max_iter=500)
df_eigenvector = to_ranked_df(eigenvector_centrality, "eigenvector_centrality")
df_eigenvector.write_parquet(config.CandidateLists().eigenvector_centrality)
df_eigenvector.head()

In [ ]:
df_eigenvector

## Next steps

Four new files now exist under `baseline_candidates/`: `pagerank.parquet`,
`personalized_pagerank.parquet`, `closeness_centrality.parquet`,
`eigenvector_centrality.parquet`. Re-run `4.comparison_candidate_set.ipynb` to score them
across the usual `k` sweep and get them into the comparison plots/tables and
`app_new_tokenizer.py` alongside the existing `highest_degree`/`most_children`/etc.
baselines -- no code changes needed there, since both use glob-based discovery over
everything in `baseline_candidates/`.